In [6]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Display settings for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [11]:
# Create dataset with Hindi sentences and their translations
dataset = {
    'sentence_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'hindi_sentence': [
        # Idioms (3)
        "उसने कल आकाश छू लिया",  # He achieved great success
        "वह हमेशा राई का पर्वत बना देता है",  # He exaggerates small things
        "मैं उस काम में अपना सिर लगा दूंगा",  # I will dedicate myself completely
        
        # Polite/Indirect requests (3)
        "क्या आप मुझे इस समस्या को हल करने में मदद कर सकते हैं?",
        "कृपया, क्या आप थोड़ी देर के लिए चुप रह सकते हैं?",
        "अगर आप को परेशानी न हो तो क्या आप यह काम कर दोगे?",
        
        # Sarcasm (2)
        "तुम तो बिल्कुल उचित समय पर पहुंच गए, सिर्फ 2 घंटे देर से!",
        "वाह, तुम्हारी गणित कौशल तो शानदार है!",
        
        # Ambiguous words (2)
        "नदी के किनारे बैंक बहुत सुंदर दिखता है",  # Bank (river bank, not financial)
        "इस खेल में हल्के और तेज दोनों प्रकार के शॉट हैं",  # Light (weight vs brightness)
    ],
    'literal_translation': [
        # Idioms
        "He touched the sky yesterday",
        "He always makes a mustard seed into a mountain",
        "I will put my head into that work",
        
        # Polite requests
        "Can you help me solve this problem?",
        "Please, can you stay quiet for a while?",
        "If it doesn't trouble you, will you do this work?",
        
        # Sarcasm
        "You arrived at the perfect time, just 2 hours late!",
        "Wow, your math skills are amazing!",
        
        # Ambiguous
        "The bank beside the river looks very beautiful",
        "This game has both light and fast types of shots",
    ],
    'pragmatic_translation': [
        # Idioms - intended meaning
        "He achieved tremendous success",
        "He exaggerates and makes small issues seem major",
        "I will dedicate myself completely to that task",
        
        # Polite requests - soften to show intent
        "Would you kindly help me solve this problem?",
        "I would appreciate it if you could keep quiet for a moment",
        "Would you be able to help me with this work, if you don't mind?",
        
        # Sarcasm - reveal the actual sentiment
        "You're late! You were supposed to be here 2 hours ago (expressing frustration)",
        "Your math skills actually need improvement (said with criticism, not praise)",
        
        # Ambiguous - disambiguated
        "The riverbank beside the river looks very beautiful",
        "This game has both lightweight and fast-moving types of shots",
    ],
    'category': [
        'idiom', 'idiom', 'idiom',
        'polite_request', 'polite_request', 'polite_request',
        'sarcasm', 'sarcasm',
        'ambiguous_word', 'ambiguous_word'
    ],
    'explanation': [
        "Idiom meaning 'achieve great success' - literal interpretation misses the actual achievement",
        "Idiom about exaggerating - literal meaning of 'mustard seed becoming mountain' obscures the intent",
        "Idiom for dedication - 'putting head' literally doesn't convey commitment",
        
        "Polite request using question form and conditional - captures politeness tone",
        "Polite request with 'please' and conditional - shows respect and indirectness",
        "Polite request using negative condition - highly indirect and respectful",
        
        "Sarcasm - praising lateness as 'perfect timing' actually expresses frustration",
        "Sarcasm - 'amazing' skills used sarcastically after poor performance implies criticism",
        
        "Word 'bank' can mean financial institution or riverbank - context clarifies it means riverbank",
        "Word 'light' can mean brightness or weight - context shows it refers to weight/intensity"
    ]
}

# Create DataFrame
df_dataset = pd.DataFrame(dataset)

print("PRAGMATIC-AWARE HINDI DATASET (10 Sentences)")
print(df_dataset.to_string())
print(f"\nDataset shape: {df_dataset.shape}")
print(f"\nCategory distribution:\n{df_dataset['category'].value_counts()}")

# Save to CSV
csv_path = 'pragmatic_hindi_dataset.csv'
df_dataset.to_csv(csv_path, index=False)
print(f"\nDataset saved to: {csv_path}")

PRAGMATIC-AWARE HINDI DATASET (10 Sentences)
   sentence_id                                             hindi_sentence                                  literal_translation                                                           pragmatic_translation        category                                                                                         explanation
0            1                                       उसने कल आकाश छू लिया                         He touched the sky yesterday                                                  He achieved tremendous success           idiom        Idiom meaning 'achieve great success' - literal interpretation misses the actual achievement
1            2                          वह हमेशा राई का पर्वत बना देता है       He always makes a mustard seed into a mountain                                He exaggerates and makes small issues seem major           idiom  Idiom about exaggerating - literal meaning of 'mustard seed becoming mountain' obscur

In [8]:
def read_pragmatic_dataset(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    return df

def extract_pragmatic_features(sentence: str, df: pd.DataFrame) -> Dict[str, bool]:
    features = {
        'is_idiom': False,
        'is_polite': False,
        'is_sarcasm': False,
        'has_ambiguous_word': False
    }
    
    # Find matching row in dataset
    matching_rows = df[df['hindi_sentence'] == sentence]
    if not matching_rows.empty:
        category = matching_rows.iloc[0]['category']
        if category == 'idiom':
            features['is_idiom'] = True
        elif category == 'polite_request':
            features['is_polite'] = True
        elif category == 'sarcasm':
            features['is_sarcasm'] = True
        elif category == 'ambiguous_word':
            features['has_ambiguous_word'] = True
    
    return features

# Read the dataset
df_data = read_pragmatic_dataset(csv_path)

print("FEATURE EXTRACTION TEST")

for idx, row in df_data.iterrows():
    sentence = row['hindi_sentence']
    features = extract_pragmatic_features(sentence, df_data)
    print(f"\nSentence {idx+1}: {sentence}")
    print(f"Features: {features}")
    print(f"Actual Category: {row['category']}")

FEATURE EXTRACTION TEST

Sentence 1: उसने कल आकाश छू लिया
Features: {'is_idiom': True, 'is_polite': False, 'is_sarcasm': False, 'has_ambiguous_word': False}
Actual Category: idiom

Sentence 2: वह हमेशा राई का पर्वत बना देता है
Features: {'is_idiom': True, 'is_polite': False, 'is_sarcasm': False, 'has_ambiguous_word': False}
Actual Category: idiom

Sentence 3: मैं उस काम में अपना सिर लगा दूंगा
Features: {'is_idiom': True, 'is_polite': False, 'is_sarcasm': False, 'has_ambiguous_word': False}
Actual Category: idiom

Sentence 4: क्या आप मुझे इस समस्या को हल करने में मदद कर सकते हैं?
Features: {'is_idiom': False, 'is_polite': True, 'is_sarcasm': False, 'has_ambiguous_word': False}
Actual Category: polite_request

Sentence 5: कृपया, क्या आप थोड़ी देर के लिए चुप रह सकते हैं?
Features: {'is_idiom': False, 'is_polite': True, 'is_sarcasm': False, 'has_ambiguous_word': False}
Actual Category: polite_request

Sentence 6: अगर आप को परेशानी न हो तो क्या आप यह काम कर दोगे?
Features: {'is_idiom': Fals

### 2.2 Rule-Based Detectors for Each Category

In [9]:
# Define idiom dictionary for Hindi idioms
HINDI_IDIOM_DICT = {
    "आकाश छू": "achieve great success",
    "राई का पर्वत": "exaggerate small issues",
    "सिर लगा": "dedicate completely",
    "हाथ धो बैठना": "give up",
    "नाक का बाल होना": "be someone's favorite",
    "टांग खींचना": "create obstacles",
    "मुंह काला करना": "bring shame",
    "कान भरना": "instigate someone"
}

# Define politeness markers in Hindi
POLITENESS_MARKERS = [
    "क्या",  # Can/Could
    "कृपया",  # Please
    "अगर",  # If
    "परेशानी",  # Trouble/bother
    "सकते",  # Can
    "दोगे",  # Will you (humble form)
    "दिजिए",  # Please give (formal)
    "कर सकते हैं",  # Can you
    "हो",  # Conditional marker
]

# Define context-dependent (ambiguous) words
AMBIGUOUS_WORDS = {
    "बैंक": ["financial bank", "river bank"],
    "हल्के": ["light (weight)", "light (brightness)"],
    "मैच": ["match (game)", "match (matchstick)"],
    "खेल": ["game", "sport"],
}

# Sarcasm patterns - often use superlatives after negative contexts
SARCASM_PATTERNS = [
    "शानदार",  # Great/Wonderful
    "बिल्कुल",  # Absolutely/Perfect
    "उचित",  # Appropriate/Perfect
    "अद्भुत",  # Amazing
    "देर से",  # Late
]

def detect_idiom(sentence: str) -> Tuple[bool, str]:
    for idiom_key, idiom_meaning in HINDI_IDIOM_DICT.items():
        if idiom_key in sentence:
            return True, idiom_key
    return False, ""

def detect_politeness(sentence: str) -> Tuple[bool, List[str]]:
    detected_markers = []
    for marker in POLITENESS_MARKERS:
        if marker in sentence:
            detected_markers.append(marker)
    
    return len(detected_markers) >= 2, detected_markers  # Need at least 2 markers

def detect_sarcasm(sentence: str, context: str = "") -> Tuple[bool, List[str]]:
    detected_patterns = []
    
    # Check for sarcasm patterns
    for pattern in SARCASM_PATTERNS:
        if pattern in sentence:
            detected_patterns.append(pattern)
    
    # Additional sarcasm indicators: superlatives followed by negative time references
    if "देर" in sentence and ("शानदार" in sentence or "बिल्कुल" in sentence):
        return True, detected_patterns
    
    return len(detected_patterns) > 0, detected_patterns

def detect_ambiguous_word(sentence: str) -> Tuple[bool, List[str]]:
    ambiguous_found = []
    for word, meanings in AMBIGUOUS_WORDS.items():
        if word in sentence:
            ambiguous_found.append(word)
    
    return len(ambiguous_found) > 0, ambiguous_found


print("DETECTOR TESTING ON ALL SENTENCES")

for idx, row in df_data.iterrows():
    sentence = row['hindi_sentence']
    actual_category = row['category']
    
    is_idiom, idiom = detect_idiom(sentence)
    is_polite, polite_markers = detect_politeness(sentence)
    is_sarcasm, sarcasm_patterns = detect_sarcasm(sentence)
    has_ambiguous, ambiguous_words = detect_ambiguous_word(sentence)
    
    print(f"\n{'─' * 100}")
    print(f"Sentence {idx+1}: {sentence}")
    print(f"Actual Category: {actual_category}")
    print(f"  └─ Idiom: {is_idiom} (detected: {idiom})")
    print(f"  └─ Politeness: {is_polite} (markers: {polite_markers})")
    print(f"  └─ Sarcasm: {is_sarcasm} (patterns: {sarcasm_patterns})")
    print(f"  └─ Ambiguous: {has_ambiguous} (words: {ambiguous_words})")

DETECTOR TESTING ON ALL SENTENCES

────────────────────────────────────────────────────────────────────────────────────────────────────
Sentence 1: उसने कल आकाश छू लिया
Actual Category: idiom
  └─ Idiom: True (detected: आकाश छू)
  └─ Politeness: False (markers: [])
  └─ Sarcasm: False (patterns: [])
  └─ Ambiguous: False (words: [])

────────────────────────────────────────────────────────────────────────────────────────────────────
Sentence 2: वह हमेशा राई का पर्वत बना देता है
Actual Category: idiom
  └─ Idiom: True (detected: राई का पर्वत)
  └─ Politeness: False (markers: [])
  └─ Sarcasm: False (patterns: [])
  └─ Ambiguous: False (words: [])

────────────────────────────────────────────────────────────────────────────────────────────────────
Sentence 3: मैं उस काम में अपना सिर लगा दूंगा
Actual Category: idiom
  └─ Idiom: True (detected: सिर लगा)
  └─ Politeness: False (markers: [])
  └─ Sarcasm: False (patterns: [])
  └─ Ambiguous: False (words: [])

───────────────────────────────

## Section 3: Translation Logic

### 3.1 Rule-Based Classifier to Choose Translation Mode

In [10]:
def classify_translation_mode(sentence: str) -> Tuple[str, Dict]:
    results = {}
    
    is_idiom, idiom = detect_idiom(sentence)
    is_polite, polite_markers = detect_politeness(sentence)
    is_sarcasm, sarcasm_patterns = detect_sarcasm(sentence)
    has_ambiguous, ambiguous_words = detect_ambiguous_word(sentence)
    
    results = {
        'is_idiom': is_idiom,
        'idiom_detected': idiom,
        'is_polite': is_polite,
        'polite_markers': polite_markers,
        'is_sarcasm': is_sarcasm,
        'sarcasm_patterns': sarcasm_patterns,
        'has_ambiguous': has_ambiguous,
        'ambiguous_words': ambiguous_words
    }
    
    if is_idiom:
        return 'idiom', results
    elif is_sarcasm:
        return 'sarcasm', results
    elif is_polite:
        return 'polite', results
    elif has_ambiguous:
        return 'ambiguous_word', results
    else:
        return 'literal', results


def translate_literal(sentence: str, df: pd.DataFrame) -> str:
    matching = df[df['hindi_sentence'] == sentence]
    if not matching.empty:
        return matching.iloc[0]['literal_translation']
    return "Translation not available"

def translate_idiom(sentence: str, df: pd.DataFrame) -> str:
    matching = df[df['hindi_sentence'] == sentence]
    if not matching.empty:
        return matching.iloc[0]['pragmatic_translation']
    return "Translation not available"

def translate_polite(sentence: str, df: pd.DataFrame) -> str:
    matching = df[df['hindi_sentence'] == sentence]
    if not matching.empty:
        translation = matching.iloc[0]['pragmatic_translation']
        # Ensure politeness markers are emphasized in translation
        return translation
    return "Translation not available"

def translate_sarcasm(sentence: str, df: pd.DataFrame) -> str:
    matching = df[df['hindi_sentence'] == sentence]
    if not matching.empty:
        translation = matching.iloc[0]['pragmatic_translation']
        # Add clarification about sarcasm
        return f"{translation} [Note: This is sarcastic - the opposite of surface meaning]"
    return "Translation not available"

def translate_ambiguous_word(sentence: str, df: pd.DataFrame) -> str:
    matching = df[df['hindi_sentence'] == sentence]
    if not matching.empty:
        translation = matching.iloc[0]['pragmatic_translation']
        return translation
    return "Translation not available"

def get_translation(sentence: str, mode: str, df: pd.DataFrame) -> str:
    translation_functions = {
        'literal': translate_literal,
        'idiom': translate_idiom,
        'polite': translate_polite,
        'sarcasm': translate_sarcasm,
        'ambiguous_word': translate_ambiguous_word
    }
    
    translator = translation_functions.get(mode, translate_literal)
    return translator(sentence, df)

print("TRANSLATION MODE CLASSIFICATION & TRANSLATION TEST")

translations_data = []

for idx, row in df_data.iterrows():
    sentence = row['hindi_sentence']
    mode, detection = classify_translation_mode(sentence)
    final_translation = get_translation(sentence, mode, df_data)
    literal = translate_literal(sentence, df_data)
    
    translations_data.append({
        'sentence_id': idx + 1,
        'hindi_sentence': sentence,
        'detected_mode': mode,
        'literal_translation': literal,
        'pragmatic_translation': final_translation,
        'actual_category': row['category'],
        'mode_match': mode == row['category']
    })
    
    print(f"\n{'─' * 100}")
    print(f"Sentence {idx+1}: {sentence}")
    print(f"Detected Mode: {mode} | Actual Category: {row['category']} | Match: {mode == row['category']}")
    print(f"Literal: {literal}")
    print(f"Pragmatic: {final_translation}")

TRANSLATION MODE CLASSIFICATION & TRANSLATION TEST

────────────────────────────────────────────────────────────────────────────────────────────────────
Sentence 1: उसने कल आकाश छू लिया
Detected Mode: idiom | Actual Category: idiom | Match: True
Literal: He touched the sky yesterday
Pragmatic: He achieved tremendous success

────────────────────────────────────────────────────────────────────────────────────────────────────
Sentence 2: वह हमेशा राई का पर्वत बना देता है
Detected Mode: idiom | Actual Category: idiom | Match: True
Literal: He always makes a mustard seed into a mountain
Pragmatic: He exaggerates and makes small issues seem major

────────────────────────────────────────────────────────────────────────────────────────────────────
Sentence 3: मैं उस काम में अपना सिर लगा दूंगा
Detected Mode: idiom | Actual Category: idiom | Match: True
Literal: I will put my head into that work
Pragmatic: I will dedicate myself completely to that task

────────────────────────────────────────